# 🩺 Dataset 3: NHANES — National Health and Nutrition Examination Survey
**Propósito:** Predicción de obesidad/sobrepeso (IMC ≥ 25) a partir de variables nutricionales y demográficas.  
**Fuente:** CDC — National Center for Health Statistics (EE. UU.).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos (formato Excel)

In [ ]:
df = pd.read_excel('../datasets/5 data sets/dataset3.xlsx')
print(f"Shape: {df.shape}")
print("Columnas:", list(df.columns))
df.head(3)

## 2. Análisis de Nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({'Nulos': nulos, '%': nulos_pct}).query('Nulos > 0').sort_values('%', ascending=False)
print(resumen)
plt.figure(figsize=(10,4))
resumen['%'].head(20).plot(kind='bar', color='tomato')
plt.title('% de valores nulos por columna')
plt.ylabel('%')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('nhanes_nulos.png', dpi=100)
plt.show()

## 3. Limpieza

In [ ]:
# Imputar numéricos con mediana, categóricos con moda
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])
df = df.drop_duplicates()
print(f"Nulos restantes: {df.isnull().sum().sum()}")
print(f"Shape limpio: {df.shape}")

## 4. EDA

In [ ]:
# Detectar columna de IMC/BMXBMI o similar
bmi_col = [c for c in df.columns if 'BMI' in c.upper() or 'BMXBMI' in c.upper()]
print("Columnas IMC detectadas:", bmi_col)

num_df = df.select_dtypes(include='number')
fig, axes = plt.subplots(1,2,figsize=(14,4))
if bmi_col:
    col = bmi_col[0]
    df[col].clip(10,60).hist(ax=axes[0], bins=40, color='steelblue', edgecolor='white')
    axes[0].set_title(f'Distribución de {col}')
    axes[0].set_xlabel('IMC (kg/m²)')
    axes[0].axvline(25, color='orange', linestyle='--', label='Sobrepeso (25)')
    axes[0].axvline(30, color='red', linestyle='--', label='Obesidad (30)')
    axes[0].legend()
corr = num_df.corr()
top = corr.abs().unstack().sort_values(ascending=False).drop_duplicates().head(20)
axes[1].axis('off')
axes[1].text(0.1, 0.95, "Top correlaciones numéricas", fontsize=12, va='top')
plt.tight_layout()
plt.savefig('nhanes_eda.png', dpi=100)
plt.show()

## 5. Preparación y Modelado

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Crear target: Sobrepeso/Obesidad si IMC >= 25
if bmi_col:
    col = bmi_col[0]
    df['TARGET'] = (df[col] >= 25).astype(int)
    X = df.drop(columns=['TARGET'] + bmi_col)
else:
    # Usar primera columna numérica como proxy si no hay IMC
    num_c = num_df.columns[0]
    df['TARGET'] = (df[num_c] > df[num_c].median()).astype(int)
    X = df.drop(columns=['TARGET', num_c])

for c in X.select_dtypes(include='object').columns:
    X[c] = le.fit_transform(X[c].astype(str))

y = df['TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")
print(f"Prevalencia sobrepeso/obesidad: {y.mean():.1%}")

In [ ]:
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred, target_names=['Normal','Sobrepeso/Obesidad']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

## ✅ Resumen
- **Target construido:** IMC ≥ 25 (sobrepeso u obesidad)
- **Modelo:** Gradient Boosting Classifier (supervisado)
- **Consideración especial NHANES:** en análisis estadístico oficial se deben usar pesos de encuesta (`WTMEC2YR`) para representatividad nacional